In [60]:
import geopandas as gpd
import pandas as pd
import numpy as np
import altair as alt

# **API Load of GeoJson Files**

Takes in files from [Sound IQ ](https://experience.arcgis.com/experience/5416ae7c51ae4244a1a42ba679c1479f/page/KELP) filtering them for the beds we care about.


Assings them to the kelp geo data frame (kelpgdf)

In [44]:
url = ('https://services.arcgis.com/6lCKYNJLvwTXqrmp/arcgis/rest/services/SIQ_Kelp_Surveys/FeatureServer/24/query?where=1%3D1&outFields=*&outSR=4326&f=geojson')
kelp_beds = gpd.read_file(url)

sites = ['Possession Point', 'Lowell Point', 'Hoypus Point', 'Polnell Point', "Ebey's Landing"]

kelpgdf = kelp_beds[kelp_beds['Location'].isin(sites)]
kelpgdf.head()

,OBJECTID,Survey_ID,County,Location,Bed_Name,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,...,ToWa,BeL,BeR,Notes,Extent_Start_Waypoint,Extent_End_Waypoint,Bulb_m2,Shape__Area,Shape__Length,geometry
0,1,Ebeys_Landing_June_14_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-06-14,14,6,2015,...,None,None,None,None,None,None,1.7590,86346.953125,1834.903661,"POLYGON ((-122.70131 48.18503, -122.70127 48.1..."
1,2,Ebeys_Landing_July_19_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-07-19,19,7,2015,...,None,None,None,None,None,None,1.8222,133068.851562,2287.171615,"POLYGON ((-122.7013 48.18498, -122.70122 48.18..."
2,3,Ebeys_Landing_Aug_15_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-08-15,15,8,2015,...,None,None,None,None,None,None,1.2000,166985.789062,2866.943303,"POLYGON ((-122.7028 48.18701, -122.7027 48.186..."
3,4,Ebeys_Landing_Sept_26_2015,Island,Ebey's Landing,Ebey's Landing,EBL,2015-09-26,26,9,2015,...,None,None,None,None,None,None,1.6889,162725.648438,2677.089037,"POLYGON ((-122.70231 48.1864, -122.70226 48.18..."
37,38,Ebeys_Landing_June_25_2016_1,Island,Ebey's Landing,Ebey's Landing,EBL,2016-06-25,25,6,2016,...,None,None,None,None,None,None,NaN,166333.468750,2196.325725,"POLYGON ((-122.699 48.18254, -122.69915 48.182..."


In [45]:
kelpgdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 60 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   OBJECTID                  131 non-null    int32         
 1   Survey_ID                 131 non-null    object        
 2   County                    131 non-null    object        
 3   Location                  131 non-null    object        
 4   Bed_Name                  131 non-null    object        
 5   site_code                 131 non-null    object        
 6   Survey_Date               131 non-null    datetime64[ms]
 7   Survey_Day                131 non-null    int32         
 8   Survey_Month              131 non-null    int32         
 9   Survey_Year               131 non-null    int32         
 10  Start_Time                126 non-null    object        
 11  End_Time                  119 non-null    object        
 12  NWSC_Max_Ext       

In [84]:
kelpgdf.value_counts('Location')

,count
Location,
Possession Point,30
Ebey's Landing,28
Hoypus Point,28
Lowell Point,23
Polnell Point,22


## **SoundIQ Data Cleaning**

We surely dont need this many columns and we dont want temperatures for all 4 data points so we can collapse those. (We may want to do something similar for depth? But depth isnt quite as complicated so.)

In [49]:
kelp_red = kelpgdf[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp',
    'Temp1_Shore_Edge',
    'Temp1_Water_Edge',
    'Temp2_Shore_Edge',
    'Temp2_Water_Edge',
    'Ave_Temp_Shore_Edge',
    'Ave_Temp_Water_Edge',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge']]

kelp_red.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Location             131 non-null    object        
 1   site_code            131 non-null    object        
 2   Survey_Date          131 non-null    datetime64[ms]
 3   Survey_Day           131 non-null    int32         
 4   Survey_Month         131 non-null    int32         
 5   Survey_Year          131 non-null    int32         
 6   Acres_NWSC_clip      131 non-null    float64       
 7   Miles                131 non-null    float64       
 8   Temp                 51 non-null     float64       
 9   Temp1_Shore_Edge     71 non-null     object        
 10  Temp1_Water_Edge     70 non-null     object        
 11  Temp2_Shore_Edge     49 non-null     object        
 12  Temp2_Water_Edge     50 non-null     object        
 13  Ave_Temp_Shore_Edge  11 non-null     flo

### Temperature Averaging from Data Foundations

In [55]:
def prepare_temp(df):

    temp_cols = [
        "Temp",
        "Ave_Temp_Shore_Edge",
        "Ave_Temp_Water_Edge",
        "Temp1_Shore_Edge",
        "Temp2_Shore_Edge",
        "Temp1_Water_Edge",
        "Temp2_Water_Edge",
    ]

    for c in temp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")


    df["Temp_Final"] = np.where(
        ~df["Temp"].isna(),
        df["Temp"],
        np.where(
            (~df["Ave_Temp_Shore_Edge"].isna()) &
            (~df["Ave_Temp_Water_Edge"].isna()),
            (df["Ave_Temp_Shore_Edge"] + df["Ave_Temp_Water_Edge"]) / 2,
            (
                ((df["Temp1_Shore_Edge"] + df["Temp2_Shore_Edge"]) / 2) +
                ((df["Temp1_Water_Edge"] + df["Temp2_Water_Edge"]) / 2)
            ) / 2
        )
    )

    return df.head()

prepare_temp(kelp_red)

/tmp/ipykernel_573/1948477887.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = pd.to_numeric(df[c], errors="coerce")
/tmp/ipykernel_573/1948477887.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Temp_Final"] = np.where(


,Location,site_code,Survey_Date,Survey_Day,Survey_Month,Survey_Year,Acres_NWSC_clip,Miles,Temp,Temp1_Shore_Edge,Temp1_Water_Edge,Temp2_Shore_Edge,Temp2_Water_Edge,Ave_Temp_Shore_Edge,Ave_Temp_Water_Edge,Depth1_Shore_Edge,Depth1_Water_Edge,Depth2_Shore_Edge,Depth2_Water_Edge,Temp_Final
0,Ebey's Landing,EBL,2015-06-14,14,6,2015,9.490754,0.760311,11.0,NaN,NaN,NaN,NaN,NaN,NaN,0.8266,1.8053,None,None,11.0
1,Ebey's Landing,EBL,2015-07-19,19,7,2015,14.626063,0.947730,13.0,NaN,NaN,NaN,NaN,NaN,NaN,0.549,1.6714,None,None,13.0
2,Ebey's Landing,EBL,2015-08-15,15,8,2015,18.354044,1.188004,13.0,NaN,NaN,NaN,NaN,NaN,NaN,0.6924,1.6775,None,None,13.0
3,Ebey's Landing,EBL,2015-09-26,26,9,2015,17.885642,1.109370,12.0,NaN,NaN,NaN,NaN,NaN,NaN,0.3965,1.7995,None,None,12.0
37,Ebey's Landing,EBL,2016-06-25,25,6,2016,18.282310,0.909986,14.0,NaN,NaN,NaN,NaN,NaN,NaN,1.525,6.1,None,None,14.0


### Slight Renaming for Clarity

In [87]:
kelp_clean= kelp_red[[
    'Location',
    'site_code',
    'Survey_Date',
    'Survey_Day',
    'Survey_Month',
    'Survey_Year',
    'Acres_NWSC_clip',
    'Miles',
    'Temp_Final',
    'Depth1_Shore_Edge',
    'Depth1_Water_Edge',
    'Depth2_Shore_Edge',
    'Depth2_Water_Edge']].rename(columns={
         'Acres_NWSC_clip' : 'area_acres',
         'Miles': 'perimeter_miles',
         'Temp_Final' : 'sstemp'
    })

kelp_clean['Location'] = kelp_clean['Location'].replace({
    "Ebey's Landing": 'Ebeys Landing'})

kelp_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Location           131 non-null    object        
 1   site_code          131 non-null    object        
 2   Survey_Date        131 non-null    datetime64[ms]
 3   Survey_Day         131 non-null    int32         
 4   Survey_Month       131 non-null    int32         
 5   Survey_Year        131 non-null    int32         
 6   area_acres         131 non-null    float64       
 7   perimeter_miles    131 non-null    float64       
 8   sstemp             99 non-null     float64       
 9   Depth1_Shore_Edge  95 non-null     object        
 10  Depth1_Water_Edge  93 non-null     object        
 11  Depth2_Shore_Edge  48 non-null     object        
 12  Depth2_Water_Edge  40 non-null     object        
dtypes: datetime64[ms](1), float64(3), int32(3), object(6)
memory usage: 12

In [88]:
kelp_clean['Location'].unique()

array(['Ebeys Landing', 'Hoypus Point', 'Lowell Point', 'Polnell Point',
       'Possession Point'], dtype=object)

### Date Time Conversion

In [58]:
kelp_clean['Survey_Date'] = pd.to_datetime(kelp_clean['Survey_Date'])
kelp_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131 entries, 0 to 556
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Location           131 non-null    object        
 1   site_code          131 non-null    object        
 2   Survey_Date        131 non-null    datetime64[ms]
 3   Survey_Day         131 non-null    int32         
 4   Survey_Month       131 non-null    int32         
 5   Survey_Year        131 non-null    int32         
 6   area_acres         131 non-null    float64       
 7   perimeter_miles    131 non-null    float64       
 8   sstemp             99 non-null     float64       
 9   Depth1_Shore_Edge  95 non-null     object        
 10  Depth1_Water_Edge  93 non-null     object        
 11  Depth2_Shore_Edge  48 non-null     object        
 12  Depth2_Water_Edge  40 non-null     object        
dtypes: datetime64[ms](1), float64(3), int32(3), object(6)
memory usage: 12

# **Locally Hosted Data Sets**

lat lon coordinates for all beds in MRC.

Conductivity, Temperature, & Dissolved Solid (CTD - Salinity/Temp) measurements from 5 beds in 2025 (Possession, Hoypus, Lowell, Ebeys (August only), Polnell (August Only).

In [72]:
locations = pd.read_csv("/content/latlon_sites.csv")
ctd_2025 = pd.read_csv("/content/kelp_bed_ctd_2025.csv")

## **Locations Cleaning**

All that needs to be done is dropping the ones we dont have the other data for, specifically ctd.

In [73]:
locations.head()

,Location,Review,lon,lat
0,Clallam Bay,No,-124.274586,48.256240
1,FWB2/Observatory Point,No,-123.637116,48.149835
2,FWB3/Extension,Review for 2024,-123.637779,48.151343
3,North Beach,No,-122.777051,48.144927
4,Edmonds Dive Park,No,-122.381880,47.816669


In [74]:
locations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Location  26 non-null     object 
 1   Review    25 non-null     object 
 2   lon       26 non-null     float64
 3   lat       26 non-null     float64
dtypes: float64(2), object(2)
memory usage: 964.0+ bytes


In [75]:
loc_sites = ['Possession Point', 'Lowell Point Camano State Park', 'Hoypus', 'Polnell', "Ebey's"]

clean_loc = locations[locations['Location'].isin(loc_sites)]

In [76]:
clean_loc['Location'] = clean_loc['Location'].replace({
    'Lowell Point Camano State Park': 'Lowell Point',
    'Hoypus': 'Hoypus Point',
    'Polnell': 'Pollnell Point',
    "Ebey's": 'Ebeys Landing'})

/tmp/ipykernel_573/189826202.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_loc['Location'] = clean_loc['Location'].replace({


In [77]:
clean_loc.head()

,Location,Review,lon,lat
10,Possession Point,No,-122.388517,47.901769
11,Ebeys Landing,No,-122.703190,48.183653
12,Pollnell Point,No,-122.565838,48.274563
13,Lowell Point,No,-122.485688,48.120390
14,Hoypus Point,No,-122.602076,48.410683


In [78]:
clean_loc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5 entries, 10 to 14
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Location  5 non-null      object 
 1   Review    5 non-null      object 
 2   lon       5 non-null      float64
 3   lat       5 non-null      float64
dtypes: float64(2), object(2)
memory usage: 200.0+ bytes


## **CTD Cleaning**

Not any cleaning neccesary (I think), was built to not need cleaning.

In [79]:
ctd_2025.head()

,site,bed,date,data_point,temp,salinity,conductivity,depth
0,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.50998,36.04,0
1,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.68870,36.23,2
2,Ebeys Landing,Bed Main,2025-08-19,DE1,12.8,30.69811,36.24,4
3,Ebeys Landing,Bed Main,2025-08-19,DE1,12.7,30.77950,36.24,6
4,Ebeys Landing,Bed Main,2025-08-19,DE1,12.7,30.78894,36.25,8


In [80]:
ctd_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 353 entries, 0 to 352
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   site          353 non-null    object 
 1   bed           353 non-null    object 
 2   date          353 non-null    object 
 3   data_point    353 non-null    object 
 4   temp          350 non-null    float64
 5   salinity      347 non-null    float64
 6   conductivity  280 non-null    float64
 7   depth         353 non-null    int64  
dtypes: float64(3), int64(1), object(4)
memory usage: 22.2+ KB


In [81]:
ctd_2025['site'].unique()

array(['Ebeys Landing', 'Hoypus Point', 'Lowell Point', 'Polnell Point',
       'Possession Point'], dtype=object)

# **EDA**

In [67]:
depth = alt.Chart(kelp_clean).transform_fold(
    ["Depth1_Shore_Edge", "Depth1_Water_Edge", "Depth2_Shore_Edge", "Depth2_Water_Edge"],
    as_=["point", "depth"]
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):T', title=None),
        alt.Y('depth:Q', title = 'Depth(ft)'),
        alt.Color('point:N'),
    ).properties(
    width=800,
    height=400)

depth

alt.Chart(...)

In [89]:
depth_ebey = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Ebeys Landing'
    ).transform_fold(
    ["Depth1_Shore_Edge", "Depth1_Water_Edge", "Depth2_Shore_Edge", "Depth2_Water_Edge"],
    as_=["point", "depth"]
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):T', title=None),
        alt.Y('depth:Q', title = 'Depth(ft)'),
        alt.Color('point:N'),
    ).properties(
    width=800,
    height=400)

depth_ebey

alt.Chart(...)

In [90]:
depth_poss = alt.Chart(kelp_clean).transform_filter(
        alt.datum.Location == 'Possession Point'
    ).transform_fold(
    ["Depth1_Shore_Edge", "Depth1_Water_Edge", "Depth2_Shore_Edge", "Depth2_Water_Edge"],
    as_=["point", "depth"]
    ).mark_line().encode(
        alt.X('yearmonth(Survey_Date):T', title=None),
        alt.Y('depth:Q', title = 'Depth(ft)'),
        alt.Color('point:N'),
    ).properties(
    width=800,
    height=400)

depth_poss

alt.Chart(...)

The missing depths are hiding in

*Ave_Depth_Shore_Edge_m       
Ave_Depth_Water_Edge_m*